# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

# Silver Transformation

In [0]:
# Clean firstname and lastname by trimming spaces
df = df.withColumn("cst_firstname", trim(col("cst_firstname"))) \
               .withColumn("cst_lastname", trim(col("cst_lastname")))

display(df)

In [0]:
df = df.filter(col("cst_id").isNotNull())

In [0]:
# Normalize marital status and gender
from pyspark.sql.functions import when

df = df.withColumn(
    "cst_marital_status",
    when(col("cst_marital_status") == "S", "Single")
    .when(col("cst_marital_status") == "M", "Married")
    .otherwise("n/a")
).withColumn(
    "cst_gndr",
    when(col("cst_gndr") == "F", "Female")
    .when(col("cst_gndr") == "M", "Male")
    .otherwise("n/a")
)

display(df)

In [0]:

rename_map = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.limit(10).display()

# Write Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

In [0]:
%sql
select * from workspace.silver.crm_customers limit 5